# GraphKV v5：第五阶段定位实验

这份 notebook 没有附带伪造的运行结果。它用于你已经配置好的 Linux/Jupyter 环境；本地研究端只检查了 notebook 格式、代码语法和不依赖模型的 mask 约束。

主要回答三个问题：LYRA 是否被其他节点干扰？公开训练格式是否影响结果？FP32 下完整回答是否仍出错？最后检查修改连边后的缓存是否过期。

建议独立启动一个 kernel，依次运行。默认单卡 BF16，物理 GPU 3；CPU FP32 生成默认关闭。请先确认下面的 `REPO_DIR`，它应直接包含 `graph_kv_cache.py` 和 `graph_kv_adapter.py`。现有依赖须为 Transformers 4.50.0。

这里的连续位置独立分块，是公开训练方式的近似对照，不是完整复刻训练数据或官方推理服务。加入 `[Block-Attention]` 的效果也只能由实验判断，不能预先当作修复。

In [1]:
from pathlib import Path
import os, sys
MODEL_PATH = Path('/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT')
REPO_DIR = Path('/home/lyh/hcy_work/GraphKV_refactor_v5')
GPU_ID = '3'
if 'torch' in sys.modules:
    raise RuntimeError('请重启 kernel，再从本格开始，确保 GPU 选择生效。')
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
RUN_CPU_FP32 = True
CPU_THREADS = 8
MAX_NEW_TOKENS = 64
OUTPUT_DIR = Path.cwd() / 'graphkv_stage5_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
assert (MODEL_PATH/'config.json').is_file(), MODEL_PATH
assert (REPO_DIR/'graph_kv_cache.py').is_file(), '请将 REPO_DIR 改为实际模块目录'
sys.path.insert(0, str(REPO_DIR))
import gc, json, hashlib, inspect, time, re
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from graph_kv_cache import GraphKVCache
from graph_kv_adapter import build_graphkv_from_ids, greedy_generate_graphkv, graphkv_next_logits
assert transformers.__version__ == '4.50.0', transformers.__version__
assert Path(inspect.getfile(GraphKVCache)).resolve().parent == REPO_DIR.resolve()
assert torch.cuda.is_available()
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH),local_files_only=True)
def tok(text):
    return tokenizer(text,return_tensors='pt',add_special_tokens=False,truncation=False).input_ids
SOURCE_TEXTS = {
    '4':'[Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.\n',
    '5':'[Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.\n'}
TARGET_TEXTS = {
    '1':'[Document 1] Project ORION uses the material supplied by ALPHA.\n',
    '2':'[Document 2] Project LYRA uses the material supplied by BETA.\n',
    '3':'[Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.\n'}
SOURCES, TARGETS = ['4','5'], ['1','2','3']
EDGES = [('4','1'),('4','3'),('5','2'),('5','3')]
BASE_PREFIX = '<|user|>\nAnswer using the following documents. Give a short factual answer.\n'
NODES = {n:tok(t) for n,t in {**SOURCE_TEXTS,**TARGET_TEXTS}.items()}
L = max(x.shape[1] for x in NODES.values())
QUESTIONS = {
    'ORION':'What material does ORION use? Answer with the material name only.',
    'LYRA':'What material does LYRA use? Answer with the material name only.',
    'NOVA':'Which two materials does NOVA combine? Answer with the material names only.',
    'BETA_fact':'What material does BETA supply? Answer with the material name only.',
    'LYRA_supplier':'Which supplier supplies the material used by LYRA? Answer with the supplier name only.'}
def query_ids(question, native=False):
    return tok('\nQuestion: '+question+('\n\n' if native else '\n')+'<|assistant|>\n')
results = {'settings':{'model_path':str(MODEL_PATH),'repo_dir':str(REPO_DIR),'gpu':GPU_ID,
    'torch':torch.__version__,'transformers':transformers.__version__, 'L':int(L),
    'max_new_tokens':MAX_NEW_TOKENS,'module_sha256':{n:hashlib.sha256((REPO_DIR/n).read_bytes()).hexdigest()
    for n in ['graph_kv_cache.py','graph_kv_adapter.py']},
    'config_sha256':hashlib.sha256((MODEL_PATH/'config.json').read_bytes()).hexdigest(),
    'source_texts':SOURCE_TEXTS,'target_texts':TARGET_TEXTS,'edges':EDGES,'prefix':BASE_PREFIX,
    'questions':QUESTIONS},'subset':[],'format':[],'fp32':[],'edge_mutation':None}
def save():
    p=OUTPUT_DIR/'graphkv_stage5_results.json'; tmp=p.with_suffix('.tmp')
    tmp.write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8'); tmp.replace(p)
def build(model,prefix_ids):
    return build_graphkv_from_ids(model=model, source_inputs={n:NODES[n] for n in SOURCES},
        target_inputs={n:NODES[n] for n in TARGETS}, edges=EDGES,prefix_input_ids=prefix_ids,chunk_span=L)
def digest(g):
    h=hashlib.sha256()
    caches=([g.prefix_cache] if g.prefix_cache is not None else [])+[g.nodes[n] for n in SOURCES+TARGETS]
    for cache in caches:
        for name in ['key_cache','value_cache']:
            for x in getattr(cache,name):
                h.update(x.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
def graph_answer(model,g,q,sources=None,targets=None):
    r=greedy_generate_graphkv(model=model,tokenizer=tokenizer,graph_cache=g,query_input_ids=q,
        source_ids=sources,target_ids=targets,max_new_tokens=MAX_NEW_TOKENS,return_details=True)
    return {'answer':r.text,'token_ids':r.token_ids,'eos':r.stopped_on_eos,
        'hit_limit':len(r.token_ids)==MAX_NEW_TOKENS and not r.stopped_on_eos}
print('已导入：',inspect.getfile(GraphKVCache),'节点长度：',{n:x.shape[1] for n,x in NODES.items()})
save()


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


已导入： /home/lyh/hcy_work/GraphKV_refactor_v5/graph_kv_cache.py 节点长度： {'4': 20, '5': 21, '1': 15, '2': 16, '3': 18}


## 1. 固定节点缓存，只改变 query 可读取的节点

所有目标先按完整图生成一次，然后固定这些缓存、问题 token 和 query 逻辑位置。逐个加入节点，避免把“重新编码了图”和“query 多读了一份缓存”混在一起。物理缓存长度和运算形状仍会变化，因此这里主要定位干扰来源，不能单独证明没有数值影响。

如果只读 source 5 + target 2 就能答银，但加入某节点后答铜，问题更接近 query 阶段的竞争。若只读这对节点也错，再看直接问 BETA 和只问 LYRA 的供应商是否答对。只读 target 2 答错，不等于边未传播；它是一个额外的读取限制。

In [2]:
model=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),local_files_only=True,
    torch_dtype=torch.bfloat16,device_map={'':0},low_cpu_mem_usage=True,attn_implementation='eager').eval()
prefix_ids=tok(BASE_PREFIX)
g=build(model,prefix_ids); before=digest(g)
q=query_ids(QUESTIONS['LYRA']); logical_start=int(g.query_position_ids(q.shape[1])[0,0])
views=[('relevant_pair',['5'],['2']),('plus_source4',['4','5'],['2']),
    ('plus_target1',['5'],['1','2']),('plus_target3',['5'],['2','3']),
    ('plus_source4_target1',['4','5'],['1','2']),('all',SOURCES,TARGETS),
    ('target2_only',[],['2'])]
for name,ss,tt in views:
    assert int(g.query_position_ids(q.shape[1])[0,0])==logical_start
    r=graph_answer(model,g,q,ss,tt)
    r.update(view=name,question='LYRA',source_ids=ss,target_ids=tt,
        query_token_ids=q[0].tolist(),query_logical_start=logical_start,
        query_physical_start=int(prefix_ids.shape[1]+sum(NODES[n].shape[1] for n in ss+tt)))
    results['subset'].append(r);save();print(name,repr(r['answer']))
for question in ['BETA_fact','LYRA_supplier']:
    for name,ss,tt in [views[0],views[5]]:
        r=graph_answer(model,g,query_ids(QUESTIONS[question]),ss,tt)
        r.update(view=name,question=question,source_ids=ss,target_ids=tt)
        results['subset'].append(r);save();print(question,name,repr(r['answer']))
assert digest(g)==before, '生成修改了保存的节点缓存'
print('PASS: 节点缓存未被查询改写。')


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.23it/s]


relevant_pair 'Silver.'
plus_source4 'Copper.'
plus_target1 'Silver.'
plus_target3 'Silver.'
plus_source4_target1 'Copper.'
all 'Copper.'
target2_only 'According to the provided documents, Project LYRA uses the material supplied by BETA, which is named in [Document 5] as "material supplied by BETA".'
BETA_fact relevant_pair 'Silver.'
BETA_fact all 'According to [Document 4], BETA supplies silver.'
LYRA_supplier relevant_pair 'BETA'
LYRA_supplier all 'According to the documents, BETA supplies the material used by LYRA.'
PASS: 节点缓存未被查询改写。


## 2. 不通过缓存拼接的独立对照

下面完整矩阵计算有三种：图注意力+共享位置、独立分块+连续位置、普通顺序。它们使用同样的分块 token，query 能读取全部前文；只有指定的 mask/位置设置不同。连续位置独立分块更接近公开训练代码，但数据格式仍可能不同。

工具逐步重新计算短上下文，适合比较内容，**不要用它的耗时评价 GraphKV 的加速比**。

In [3]:
def layout(prefix_ids,q,attention='graph',positions='shared'):
    chunks=[('prefix',prefix_ids)]+[(n,NODES[n]) for n in SOURCES+TARGETS]+[('query',q)]
    ranges={}; pos=[]; ids=[]; offset=0; P=int(prefix_ids.shape[1])
    for name,x in chunks:
        n=int(x.shape[1]);ranges[name]=(offset,offset+n)
        start=offset if positions=='continuous' else (0 if name=='prefix' else
            P if name in SOURCES else P+L if name in TARGETS else P+2*L)
        pos.extend(range(start,start+n));offset+=n;ids.append(x.cpu())
    allow=torch.zeros((offset,offset),dtype=torch.bool)
    if attention=='causal':allow=torch.ones_like(allow).tril()
    else:
        for name,(a,b) in ranges.items():
            allow[a:b,a:b]=torch.ones((b-a,b-a),dtype=torch.bool).tril()
            if name=='query':allow[a:b,:a]=True
            elif attention=='graph' and name in TARGETS:
                for s,t in EDGES:
                    if t==name:
                        c,d=ranges[s];allow[a:b,c:d]=True
    return torch.cat(ids,-1),torch.tensor([pos]),allow,ranges
@torch.inference_mode()
def dense_logits(model,prefix_ids,q,attention='graph',positions='shared'):
    ids,pos,allow,_=layout(prefix_ids,q,attention,positions)
    dev=model.get_input_embeddings().weight.device; dtype=next(model.parameters()).dtype
    mask=torch.zeros((1,1,ids.shape[1],ids.shape[1]),dtype=dtype,device=dev)
    mask.masked_fill_(~allow.to(dev)[None,None],torch.finfo(dtype).min)
    out=model(ids.to(dev),position_ids=pos.to(dev),cache_position=torch.arange(ids.shape[1],device=dev),
        attention_mask=mask,use_cache=False,return_dict=True)
    return out.logits[:,-1].detach().float()
@torch.inference_mode()
def dense_answer(model,prefix_ids,q,attention,positions):
    q=q.cpu();tokens=[];stopped=False
    eos=model.generation_config.eos_token_id
    if eos is None:eos=tokenizer.eos_token_id
    eos=set([] if eos is None else [eos] if isinstance(eos,int) else eos)
    for _ in range(MAX_NEW_TOKENS):
        token=int(dense_logits(model,prefix_ids,q,attention,positions).argmax(-1).item())
        tokens.append(token)
        if token in eos:stopped=True;break
        q=torch.cat([q,torch.tensor([[token]],dtype=torch.long)],-1)
    return {'answer':tokenizer.decode(tokens,skip_special_tokens=True),'token_ids':tokens,
        'eos':stopped,'hit_limit':len(tokens)==MAX_NEW_TOKENS and not stopped}
_,pos,a,r=layout(prefix_ids,query_ids(QUESTIONS['LYRA']))
for t in TARGETS:
    x,y=r[t]
    for s in SOURCES:
        u,v=r[s];assert bool(a[x:y,u:v].all())==((s,t) in EDGES)
    u,v=r['prefix'];assert not a[x:y,u:v].any()
x,y=r['query'];assert a[x:y,:x].all()
assert pos[0,x].item()==prefix_ids.shape[1]+2*L
_,pc,ac,rc=layout(prefix_ids,query_ids(QUESTIONS['LYRA']),'independent','continuous')
assert torch.equal(pc,torch.arange(pc.shape[1]).unsqueeze(0))
for t in TARGETS:
    x,y=rc[t]
    for s in SOURCES:
        u,v=rc[s];assert not ac[x:y,u:v].any()
print('PASS: 图 mask、独立分块 mask、query 可见范围及位置检查。')


PASS: 图 mask、独立分块 mask、query 可见范围及位置检查。


## 3. 模板与训练标记对照

比较 legacy/native 换行和标记有/无，共四种输入。分块模式加标记时将其放在整个输入最前，与公开训练代码一致；prefix 长度因此改变，后续逻辑位置会同步偏移。这是一个“训练格式整体变化”的测试，不能把所有效果只归因于标记文字。

普通顺序对照也保留同样的输入标记，方便判断变化是否只在分块模式发生；这并不声称公开训练的普通顺序样本加过该标记。每格仅回答 ORION、LYRA 两题，避免用一次联合回答掩盖单题错误。图条件使用 v5，独立分块与顺序条件使用完整矩阵。

In [4]:
for marker in [False,True]:
    p=tok(('[Block-Attention]' if marker else '')+BASE_PREFIX)
    fg=g if not marker else build(model,p)
    initial=digest(fg)
    for native in [False,True]:
        for question in ['ORION','LYRA']:
            q=query_ids(QUESTIONS[question],native=native)
            for condition in ['v5_graph_shared','independent_continuous','ordinary_sequence']:
                if condition=='v5_graph_shared':r=graph_answer(model,fg,q)
                else:r=dense_answer(model,p,q,'independent' if condition=='independent_continuous' else 'causal','continuous')
                r.update(marker=marker,native=native,question=question,condition=condition,
                    prefix_token_ids=p[0].tolist(),query_token_ids=q[0].tolist(),P=int(p.shape[1]))
                results['format'].append(r);save();print(marker,native,question,condition,repr(r['answer']))
    assert digest(fg)==initial
    if marker:del fg
print('完成格式对照；请同时看材料、供应商和完整回答。')


False False ORION v5_graph_shared 'Copper.'
False False ORION independent_continuous 'Copper.'
False False ORION ordinary_sequence 'Copper'
False False LYRA v5_graph_shared 'Copper.'
False False LYRA independent_continuous 'According to Document 3, LYRA uses the material supplied by BETA, which is silver.'
False False LYRA ordinary_sequence 'According to Document 2, Project LYRA uses the material supplied by BETA, which is silver.'
False True ORION v5_graph_shared 'Copper.'
False True ORION independent_continuous 'According to Document 1, ORION uses the material supplied by ALPHA, which is copper.'
False True ORION ordinary_sequence 'Copper'
False True LYRA v5_graph_shared 'Copper.'
False True LYRA independent_continuous 'According to Document 3, LYRA uses the material supplied by BETA, which is silver.'
False True LYRA ordinary_sequence 'According to Document 2, Project LYRA uses the material supplied by BETA, which is silver.'
True False ORION v5_graph_shared 'Copper.'
True False ORI

## 4. 修改连边是否让已有缓存失效

这格在已构建图上把 target 2 的依赖从 source 5 改成 source 4，然后检查缓存是否变动。结束后恢复边，不影响后续实验。现有 v5 接口预计允许修改边但保留旧缓存；这是生命周期隐患，不能由此认定前面静态构建的图回答错因就在这里。

若你的代码已增加“编码后禁止改边”的保护，此处拒绝操作是预期的安全行为。

In [5]:
original_edges=[(s,t) for t,ss in g.graph.items() for s in ss]
old=digest(g); status={}
try:
    changed=[('4','1'),('4','3'),('4','2'),('5','3')]
    g.set_edges(changed)
    status={'edit_allowed':True,'new_target2_dependencies':g.source_nodes_for('2'),
        'cached_bytes_unchanged':digest(g)==old}
    try:g.validate();status['validate_accepts']=True
    except Exception as ex:status['validate_accepts']=False;status['validation_error']=str(ex)
except Exception as ex:
    status={'edit_allowed':False,'error':str(ex)}
finally:
    if status.get('edit_allowed'):g.set_edges(original_edges)
results['edge_mutation']=status;save();print(status)
assert digest(g)==old
del g, model
gc.collect();torch.cuda.empty_cache()


{'edit_allowed': True, 'new_target2_dependencies': ['4'], 'cached_bytes_unchanged': True, 'validate_accepts': True}


## 5. 可选 CPU FP32：比较完整生成

第四阶段 FP32 只比较了缓存和首个 token 的分数，没有完整回答。这格补齐 ORION、LYRA、NOVA 各题的 v5 与完整图生成，并对比同一个 CPU FP32 模型的首个 token 分数。

默认跳过；在首格设 `RUN_CPU_FP32=True` 才执行。8B FP32 权重约 30 GiB，还需加载与运行空间；这里要求当前可用 RAM 至少 48 GiB。CPU 生成可能较慢。对比先前单卡 BF16 时设备和精度一起变化，仍不能独立估计精度的因果作用。FP32 只让计算更精细，不会恢复已保存在 BF16 权重中丢失的精度。

In [6]:
if RUN_CPU_FP32:
    import psutil
    available=psutil.virtual_memory().available/2**30
    assert available>=48, f'当前可用 RAM {available:.1f} GiB，暂不运行 CPU FP32'
    torch.set_num_threads(CPU_THREADS)
    cpu_model=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),local_files_only=True,
        torch_dtype=torch.float32,device_map={'':'cpu'},low_cpu_mem_usage=True,attn_implementation='eager').eval()
    cp=tok(BASE_PREFIX);cg=build(cpu_model,cp);initial=digest(cg)
    for question in ['ORION','LYRA','NOVA']:
        q=query_ids(QUESTIONS[question]);t=time.perf_counter()
        vl=graphkv_next_logits(model=cpu_model,graph_cache=cg,query_input_ids=q)
        dl=dense_logits(cpu_model,cp,q)
        v=graph_answer(cpu_model,cg,q);d=dense_answer(cpu_model,cp,q,'graph','shared')
        row={'question':question,'v5':v,'dense_graph':d,
            'first_logit_max_diff':float((vl-dl).abs().max()),
            'v5_first_argmax':int(vl.argmax(-1).item()),'dense_first_argmax':int(dl.argmax(-1).item()),
            'same_generated_tokens':v['token_ids']==d['token_ids'],'seconds':time.perf_counter()-t}
        results['fp32'].append(row);save();print(question,repr(v['answer']),repr(d['answer']))
    assert digest(cg)==initial
    del cg,cpu_model;gc.collect()
else:
    results['fp32_status']='skipped_by_user_setting';save();print('CPU FP32 已跳过。')
print('结果保存到：',OUTPUT_DIR/'graphkv_stage5_results.json')


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.24it/s]


ORION 'Copper.' 'Copper.'
LYRA 'Copper.' 'Copper.'
NOVA 'According to the provided documents, Project NOVA combines copper supplied by ALPHA and silver supplied by BETA.' 'According to the provided documents, Project NOVA combines copper supplied by ALPHA and silver supplied by BETA.'
结果保存到： /home/lyh/hcy_work/GraphKV_refactor_v5/graphkv_stage5_outputs/graphkv_stage5_results.json


## 如何解读

- **相关两节点正确，加某节点后错误**：优先研究 query 读取竞争；不要马上修改目标节点传播公式。
- **直接问 BETA 正确，问 LYRA 的供应商也正确，但 LYRA 的材料错**：错误更接近把两条事实接起来的环节。
- **独立分块连续位置正确、图共享位置错**：图机制或位置安排改变了模型适应程度；二者尚需继续拆开，不能凭这一格独断。
- **训练标记仅使分块模式改善**：支持输入格式适配这一方向；它仍需更多独立事实组合验证。
- **CPU FP32 下 v5 与完整图完整生成相同且仍答错**：显著削弱“缓存实现算错造成语义错误”的解释。
- **CPU FP32 答对**：精度/设备因素值得继续核查；下一步应在同样的多 GPU 层分配下对比 BF16/FP32。
- **改边后缓存不变，validate 仍通过**：不要在已编码对象上直接改边；新建图，或为类增加缓存失效/禁止修改保护。

不要将这些少量定位问题当作论文基准成绩。所有回答保留原文；出现解释时，要检查每个“供应商—材料”配对，而不只是看有没有铜、银两个词。